# 04 · Continuous metrics — `foe.continuous`

**Hexkit surfaces:** `utility/automation_engine.py` (`run_comparison_suite`, `winsorize_series`),
`pages/user_level_analysis.py` (`run_business_case`; the page's own tests and outlier handling
are moving to `run_comparison_suite`, `detect_outliers` and `winsorize_series`), `pages/pre_test_analysis.py` (`fit_gamma`, `fit_negbin`).

| FOE function | R reference |
|---|---|
| `run_comparison_suite`, heuristic path: normality gate (D'Agostino K² on OLS residuals), Levene (median) gate, then ANOVA / Welch / Kruskal-Wallis / Mann-Whitney | `fBasics::dagoTest`, `car::leveneTest(center = median)`, `anova(lm)`, `oneway.test(var.equal = FALSE)`, `kruskal.test`, `wilcox.test` (exact/asymptotic chosen with scipy's rule) |
| heuristic path: global statistic (F, H, U) and effect size (η², η²_H, rank-biserial) | `anova(lm)` sums of squares, `oneway.test`, `kruskal.test`, `rstatix::kruskal_effsize`, `wilcox.test` W |
| native `welch_anova`, `tukey_hsd`, `games_howell`, `dunn_test` (post-hocs for ANOVA / Welch / Kruskal-Wallis) | `oneway.test(var.equal = FALSE)`, `TukeyHSD(aov)`, `rstatix::games_howell_test`, `dunn.test(method = "bonferroni", altp = TRUE)` |
| heuristic path, count-KPI gate → Negative Binomial LRT | `MASS::glm.nb` (full vs intercept-only), `pchisq` |
| gamma path, per transaction: Gamma LRT + Bonferroni post-hoc | exact Gamma MLE (score equation `log k − ψ(k) = log x̄ − mean(log x)` solved with `uniroot`), `pchisq` |
| gamma path, per visitor: two-part hurdle LRT | Bernoulli log-likelihood + exact Gamma MLE on the positives |
| `fit_gamma`, `fit_negbin` | exact Gamma MLE and `MASS::fitdistr(x, "gamma")`; `MASS::fitdistr(x, "negative binomial")` |
| `winsorize_series` (incl. `exclude_zeros`, `"Central Percentile"`) | `mean ± k·sd`, `quantile(type = 7)`, on the non-zero values when zeros are excluded |
| `detect_outliers` (OLS influence; per-variant SD for large data) | `rstandard`, `hatvalues`, `dffits` on `lm`; `mean ± k·sd` per variant |
| `run_business_case` (delta method) | `msm::deltamethod` |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.core.models import (AlternativeHypothesis, AnalysisUnit, ContinuousApproach,
                             ContinuousMetricConfig)
from foe.continuous.operations import ContinuousMetricEngine
from validation_helpers import Checks

checks = Checks()
engine = ContinuousMetricEngine()
rng = np.random.default_rng(2026)


def make_df(groups):
    """groups: {label: array} -> long DataFrame with columns variant, value."""
    return pd.DataFrame([(g, float(v)) for g, arr in groups.items() for v in arr],
                        columns=["variant", "value"])

## 1. Heuristic decision tree

One dataset per branch. The R function below re-implements the same decision tree using R's
own tests, so the comparison covers the gates (`is_normal`, `is_homogeneous`), the chosen
test and its p-value.

In [2]:
datasets = {
    "normal, equal var (ANOVA)": make_df({
        "A": rng.normal(50, 10, 400), "B": rng.normal(51.5, 10, 400), "C": rng.normal(50.5, 10, 400)}),
    "normal, unequal var (Welch)": make_df({
        "A": rng.normal(50, 5, 400), "B": rng.normal(51, 12, 400), "C": rng.normal(50, 20, 400)}),
    "skewed, 2 groups (Mann-Whitney)": make_df({
        "A": rng.lognormal(3.5, 0.9, 500), "B": rng.lognormal(3.6, 0.9, 520)}),
    "skewed, 3 groups (Kruskal-Wallis)": make_df({
        "A": rng.lognormal(3.5, 0.9, 400), "B": rng.lognormal(3.6, 0.9, 400), "C": rng.lognormal(3.45, 0.9, 400)}),
    "small sample (<20 rows, non-parametric)": make_df({
        "A": rng.normal(10, 2, 8), "B": rng.normal(12, 2, 9)}),
}

heur_py = []
for name, df in datasets.items():
    res = engine.run_comparison_suite(df, ContinuousMetricConfig(
        kpi="value", group_col="variant", approach=ContinuousApproach.HEURISTIC,
        unit=AnalysisUnit.PER_TRANSACTION, control_label="A"))
    heur_py.append(dict(name=name, test=res.test_name, p_value=res.p_value,
                        is_normal=res.is_normal, is_homogeneous=res.is_homogeneous,
                        statistic=res.test_statistic, effect_size=res.effect_size))
heur_py = pd.DataFrame(heur_py)
heur_py

,name,test,p_value,is_normal,is_homogeneous,statistic,effect_size
0,"normal, equal var (ANOVA)",Standard ANOVA,0.009942,True,True,4.628834,0.007675
1,"normal, unequal var (Welch)",Kruskal-Wallis,0.977833,False,False,0.044832,0.000000
2,"skewed, 2 groups (Mann-Whitney)",Mann-Whitney U,0.009730,False,True,117840.000000,0.093538
3,"skewed, 3 groups (Kruskal-Wallis)",Kruskal-Wallis,0.470354,False,False,1.508539,0.000000
4,"small sample (<20 rows, non-parametric)",Mann-Whitney U,0.113945,False,True,19.000000,0.472222


In [3]:
%%R
suppressPackageStartupMessages({library(fBasics); library(car); library(MASS); library(msm); library(rstatix)})

heuristic_r <- function(df) {
  df$variant <- factor(df$variant)
  fit <- lm(value ~ variant, data = df)
  res <- residuals(fit)
  is_normal <- if (length(res) >= 20) dagoTest(res)@test$p.value[1] >= 0.05 else FALSE
  is_homog  <- leveneTest(value ~ variant, data = df, center = median)[1, "Pr(>F)"] >= 0.05
  k <- nlevels(df$variant)
  ss <- anova(fit)[, "Sum Sq"]
  if (is_normal && is_homog) {
    test <- "Standard ANOVA"; p <- anova(fit)[1, "Pr(>F)"]
    stat <- anova(fit)[1, "F value"]; eff <- ss[1] / sum(ss)
  } else if (is_normal) {
    ow <- oneway.test(value ~ variant, data = df, var.equal = FALSE)
    test <- "Welch's ANOVA"; p <- ow$p.value; stat <- unname(ow$statistic); eff <- ss[1] / sum(ss)
  } else if (k > 2) {
    kt <- kruskal.test(value ~ variant, data = df)
    test <- "Kruskal-Wallis"; p <- kt$p.value; stat <- unname(kt$statistic)
    eff <- kruskal_effsize(df, value ~ variant)$effsize
  } else {
    g <- split(df$value, df$variant)
    test <- "Mann-Whitney U"
    # scipy's method="auto": exact if either arm has <= 8 rows and there are no ties
    exact <- min(lengths(g)) <= 8 && !any(duplicated(df$value))
    wt <- wilcox.test(g[[1]], g[[2]], exact = exact, correct = TRUE)
    p <- wt$p.value; stat <- unname(wt$statistic)  # W = U of the first group
    eff <- 1 - 2 * stat / (length(g[[1]]) * length(g[[2]]))
  }
  data.frame(test = test, p_value = p, is_normal = is_normal, is_homogeneous = is_homog,
             statistic = stat, effect_size = eff)
}

In addition: Warning messages:
1: package 'fBasics' was built under R version 4.6.2 
2: package 'car' was built under R version 4.6.2 
3: package 'carData' was built under R version 4.6.2 
4: package 'msm' was built under R version 4.6.2 
5: package 'rstatix' was built under R version 4.6.2 


In [4]:
heur_r = []
for name, df in datasets.items():
    %R -i df -o out out <- heuristic_r(df)
    heur_r.append(out.iloc[0].to_dict() | {"name": name})
heur_r = pd.DataFrame(heur_r)
heur_r

,test,p_value,is_normal,is_homogeneous,statistic,effect_size,name
0,Standard ANOVA,0.009942,True,True,4.628834,0.007675,"normal, equal var (ANOVA)"
1,Kruskal-Wallis,0.977833,False,False,0.044832,0.000000,"normal, unequal var (Welch)"
2,Mann-Whitney U,0.009730,False,True,117840.000000,0.093538,"skewed, 2 groups (Mann-Whitney)"
3,Kruskal-Wallis,0.470354,False,False,1.508539,0.000000,"skewed, 3 groups (Kruskal-Wallis)"
4,Mann-Whitney U,0.113945,False,True,19.000000,0.472222,"small sample (<20 rows, non-parametric)"


In [5]:
for i, row in heur_py.iterrows():
    r = heur_r.iloc[i]
    checks.add_equal(f"is_normal       {row['name']}", bool(row.is_normal), bool(r.is_normal))
    checks.add_equal(f"is_homogeneous  {row['name']}", bool(row.is_homogeneous), bool(r.is_homogeneous))
    checks.add_equal(f"test chosen     {row['name']}", row.test, r.test)
    checks.add(f"p-value         {row['name']}", row.p_value, r.p_value, atol=1e-10, rtol=1e-6)
    checks.add(f"statistic       {row['name']}", row.statistic, r.statistic, rtol=1e-8)
    checks.add(f"effect size     {row['name']}", row.effect_size, r.effect_size, atol=1e-10, rtol=1e-8)

## 1b. Native Welch's ANOVA and post-hocs

FOE implements Welch's ANOVA and the three heuristic-path post-hocs itself (numpy plus
`scipy.stats.studentized_range` / `norm`), instead of depending on pingouin and scikit-posthocs.
Each is checked on every pair of three fixed datasets, whether or not the global test is
significant:

- **Tukey HSD** (Tukey-Kramer for unequal n) vs `TukeyHSD(aov(...))`: mean difference, CI, p.
- **Games-Howell** vs `rstatix::games_howell_test(detailed = TRUE)`: mean difference, t, CI.
  rstatix rounds its adjusted p-value to 3 decimals, so the unrounded p is recomputed from its
  t and df with R's `ptukey`.
- **Dunn** (tie-corrected, Bonferroni) vs `dunn.test(method = "bonferroni", altp = TRUE)`:
  z, raw and adjusted p.

FOE reports each pair as "later label vs earlier label" with
`mean_diff = mean(later) − mean(earlier)`, which is `TukeyHSD`'s `B-A` convention.

In [6]:
E = ContinuousMetricEngine
rng_ph = np.random.default_rng(2027)  # own stream: later sections keep their data
ph_sets = {
    "3 groups, equal var": make_df({"A": rng_ph.normal(50, 10, 60), "B": rng_ph.normal(55, 10, 45),
                                    "C": rng_ph.normal(52, 10, 70)}),
    "4 groups, unequal var": make_df({"A": rng_ph.normal(50, 4, 30), "B": rng_ph.normal(53, 9, 50),
                                      "C": rng_ph.normal(50, 15, 25), "D": rng_ph.normal(56, 6, 40)}),
    "skewed with ties": make_df({"A": np.round(rng_ph.lognormal(3.0, 0.8, 40)),
                                 "B": np.round(rng_ph.lognormal(3.3, 0.8, 55)),
                                 "C": np.round(rng_ph.lognormal(3.1, 0.8, 35))}),
}
ph_py, welch_py = [], []
for name, df in ph_sets.items():
    grouped = list(df.groupby("variant"))
    labels, groups = [g for g, _ in grouped], [x.value.to_numpy() for _, x in grouped]
    f, df1, df2, p = E.welch_anova(groups)
    welch_py.append(dict(set=name, F=f, df2=df2, p_value=p))
    for method, fn in [("tukey", E.tukey_hsd), ("games-howell", E.games_howell),
                       ("dunn", E.dunn_test)]:
        for r in fn(labels, groups):
            ph_py.append(dict(set=name, method=method, comparison=r.comparison,
                              statistic=r.statistic, p_value=r.p_value, p_adj=r.p_adj,
                              mean_diff=r.mean_diff, ci_low=r.ci_low, ci_high=r.ci_high))
ph_py = pd.DataFrame(ph_py)
ph_py

,set,method,comparison,statistic,p_value,p_adj,mean_diff,ci_low,ci_high
0,"3 groups, equal var",tukey,B vs A,5.177099,0.000971,0.000971,7.690861,2.723976,12.657745
1,"3 groups, equal var",tukey,C vs A,4.377922,0.006456,0.006456,5.802173,1.371004,10.233343
2,"3 groups, equal var",tukey,C vs B,1.312172,0.623455,0.623455,-1.888687,-6.701123,2.923749
3,"3 groups, equal var",games-howell,B vs A,3.734157,0.000915,0.000915,7.690861,2.789522,12.592199
4,"3 groups, equal var",games-howell,C vs A,3.043259,0.007980,0.007980,5.802173,1.279772,10.324575
5,"3 groups, equal var",games-howell,C vs B,-0.951346,0.609248,0.609248,-1.888687,-6.613024,2.835650
6,"3 groups, equal var",dunn,B vs A,3.613903,0.000302,0.000905,NaN,NaN,NaN
7,"3 groups, equal var",dunn,C vs A,3.477736,0.000506,0.001517,NaN,NaN,NaN
8,"3 groups, equal var",dunn,C vs B,-0.527676,0.597724,1.000000,NaN,NaN,NaN
9,"4 groups, unequal var",tukey,B vs A,2.977925,0.156293,0.156293,4.724174,-1.108792,10.557141


In [7]:
%%R
suppressPackageStartupMessages({library(rstatix); library(dunn.test)})

posthoc_r <- function(df) {
  df$variant <- factor(df$variant)
  k <- nlevels(df$variant)
  ow <- oneway.test(value ~ variant, data = df, var.equal = FALSE)
  welch <- data.frame(F = unname(ow$statistic), df2 = unname(ow$parameter[2]),
                      p_value = ow$p.value)

  tk <- TukeyHSD(aov(value ~ variant, data = df))$variant     # rows "B-A": mean(B) - mean(A)
  tukey <- data.frame(method = "tukey", comparison = sub("-", " vs ", rownames(tk)),
                      statistic = NA, p_value = tk[, "p adj"], p_adj = tk[, "p adj"],
                      mean_diff = tk[, "diff"], ci_low = tk[, "lwr"], ci_high = tk[, "upr"])

  gh <- as.data.frame(games_howell_test(df, value ~ variant, detailed = TRUE))
  # estimate = mean(group2) - mean(group1). rstatix's se is sqrt((s1^2/n1 + s2^2/n2) / 2),
  # so estimate / se is the studentized-range q; Welch's t = q / sqrt(2).
  t_gh <- gh$estimate / (gh$se * sqrt(2))
  games <- data.frame(method = "games-howell", comparison = paste(gh$group2, "vs", gh$group1),
                      statistic = t_gh,
                      p_value = ptukey(sqrt(2) * abs(t_gh), k, gh$df, lower.tail = FALSE),
                      p_adj = gh$p.adj, mean_diff = gh$estimate,
                      ci_low = gh$conf.low, ci_high = gh$conf.high)

  invisible(capture.output(dt <- dunn.test(df$value, df$variant, method = "bonferroni",
                                           altp = TRUE, kw = FALSE, table = FALSE)))
  parts <- strsplit(dt$comparisons, " - ")                    # "A - B": Z = rank(A) - rank(B)
  dunn <- data.frame(method = "dunn",
                     comparison = sapply(parts, function(x) paste(x[2], "vs", x[1])),
                     statistic = -dt$Z, p_value = dt$altP, p_adj = dt$altP.adjusted,
                     mean_diff = NA, ci_low = NA, ci_high = NA)
  list(welch = welch, posthoc = rbind(tukey, games, dunn))
}

In addition: Warning message:
package 'dunn.test' was built under R version 4.6.2 


In [8]:
ph_r, welch_r = [], []
for name, df in ph_sets.items():
    %R -i df -o out_ph -o out_w res <- posthoc_r(df); out_ph <- res$posthoc; out_w <- res$welch
    ph_r.append(out_ph.assign(set=name))
    welch_r.append(out_w.iloc[0].to_dict() | {"set": name})
ph_r = pd.concat(ph_r, ignore_index=True)
welch_r = pd.DataFrame(welch_r)
ph_r



FWER = 0.05
Reject Ho if adjusted p = FWER, where (unadjusted) p = Pr(|Z| = |z|)




FWER = 0.05
Reject Ho if adjusted p = FWER, where (unadjusted) p = Pr(|Z| = |z|)




FWER = 0.05
Reject Ho if adjusted p = FWER, where (unadjusted) p = Pr(|Z| = |z|)


,method,comparison,statistic,p_value,p_adj,mean_diff,ci_low,ci_high,set
0,tukey,B vs A,NaN,0.000971,0.000971,7.690861,2.723976,12.657745,"3 groups, equal var"
1,tukey,C vs A,NaN,0.006456,0.006456,5.802173,1.371004,10.233343,"3 groups, equal var"
2,tukey,C vs B,NaN,0.623455,0.623455,-1.888687,-6.701123,2.923749,"3 groups, equal var"
3,games-howell,B vs A,3.734157,0.000915,0.000915,7.690861,2.789522,12.592199,"3 groups, equal var"
4,games-howell,C vs A,3.043259,0.007980,0.007980,5.802173,1.279772,10.324575,"3 groups, equal var"
5,games-howell,C vs B,-0.951346,0.609248,0.609248,-1.888687,-6.613024,2.835650,"3 groups, equal var"
6,dunn,B vs A,3.613903,0.000302,0.000905,NaN,NaN,NaN,"3 groups, equal var"
7,dunn,C vs A,3.477736,0.000506,0.001517,NaN,NaN,NaN,"3 groups, equal var"
8,dunn,C vs B,-0.527676,0.597724,1.000000,NaN,NaN,NaN,"3 groups, equal var"
9,tukey,B vs A,NaN,0.156293,0.156293,4.724174,-1.108792,10.557141,"4 groups, unequal var"


In [9]:
for py, r in zip(welch_py, welch_r.to_dict("records")):
    checks.add(f"Welch F         {py['set']}", py["F"], r["F"], rtol=1e-10)
    checks.add(f"Welch df2       {py['set']}", py["df2"], r["df2"], rtol=1e-10)
    checks.add(f"Welch p         {py['set']}", py["p_value"], r["p_value"], atol=1e-12, rtol=1e-8)

merged = ph_py.merge(ph_r, on=["set", "method", "comparison"], suffixes=("", "_r"),
                     how="outer", indicator=True)
checks.add_equal("post-hoc pairs: same set in FOE and R", int((merged._merge != "both").sum()), 0)
for _, m in merged.iterrows():
    tag = f"{m.method:12s} {m.set} {m.comparison}"
    checks.add(f"p_value     {tag}", m.p_value, m.p_value_r, atol=1e-8, rtol=1e-5)
    if m.method == "games-howell":   # rstatix rounds p.adj to 3 decimals
        checks.add(f"p_adj (3dp) {tag}", m.p_adj, m.p_adj_r, atol=5e-4 + 1e-9)
    else:
        checks.add(f"p_adj       {tag}", m.p_adj, m.p_adj_r, atol=1e-8, rtol=1e-5)
    if m.method != "tukey":
        checks.add(f"statistic   {tag}", m.statistic, m.statistic_r, rtol=1e-8)
    if m.method != "dunn":
        for col in ["mean_diff", "ci_low", "ci_high"]:
            checks.add(f"{col:11s} {tag}", m[col], m[f"{col}_r"], atol=1e-10, rtol=1e-6)

## 2. Count KPI → Negative Binomial likelihood-ratio test

Integer, low-cardinality KPIs (items per order, tickets…) skip the decision tree and go to an
NB2 regression LRT. R: `glm.nb(y ~ variant)` vs `glm.nb(y ~ 1)`, dispersion α = 1/θ.

Note: with `unit = per_transaction` (the config default) FOE drops zero rows *before* the
test, so the NB is fit to the positive counts only. That is right for "items per order"
(an order always has ≥ 1 item) but not for a per-visitor count with real zeros; both units are
checked here and R applies the same zero filter.

In [10]:
def nb_draw(mean, alpha, n):
    size = 1 / alpha
    return rng.negative_binomial(size, size / (size + mean), n)

count_sets = {
    "counts, 2 groups": make_df({"A": nb_draw(2.0, 0.6, 600), "B": nb_draw(2.3, 0.6, 600)}),
    "counts, 3 groups": make_df({"A": nb_draw(3.0, 0.4, 500), "B": nb_draw(3.0, 0.4, 500),
                                 "C": nb_draw(3.4, 0.4, 500)}),
}
nb_py = []
for name, df in count_sets.items():
    for unit in [AnalysisUnit.PER_VISITOR, AnalysisUnit.PER_TRANSACTION]:
        res = engine.run_comparison_suite(df, ContinuousMetricConfig(
            kpi="value", group_col="variant", approach=ContinuousApproach.HEURISTIC,
            unit=unit, control_label="A"))
        nb_py.append(dict(name=f"{name} | {unit.value}", unit=unit.value, data=name,
                          test=res.test_name, p_value=res.p_value, alpha=res.dispersion_alpha))
nb_py = pd.DataFrame(nb_py)
nb_py

,name,unit,data,test,p_value,alpha
0,"counts, 2 groups | per_visitor",per_visitor,"counts, 2 groups",Negative Binomial Regression (LRT),0.059802,0.669783
1,"counts, 2 groups | per_transaction",per_transaction,"counts, 2 groups",Negative Binomial Regression (LRT),0.018272,0.174781
2,"counts, 3 groups | per_visitor",per_visitor,"counts, 3 groups",Negative Binomial Regression (LRT),0.145331,0.436171
3,"counts, 3 groups | per_transaction",per_transaction,"counts, 3 groups",Negative Binomial Regression (LRT),0.196138,0.194951


In [11]:
nb_r = []
for _, row in nb_py.iterrows():
    df, unit = count_sets[row.data], row.unit
    %R -i df -i unit -o out if (unit == 'per_transaction') df <- df[df$value != 0, ]; df$variant <- factor(df$variant); m1 <- glm.nb(value ~ variant, data = df); m0 <- glm.nb(value ~ 1, data = df); lr <- 2 * (as.numeric(logLik(m1)) - as.numeric(logLik(m0))); out <- data.frame(p_value = pchisq(lr, df = nlevels(df$variant) - 1, lower.tail = FALSE), alpha = 1 / m1$theta)
    nb_r.append(out.iloc[0].to_dict())
nb_r = pd.DataFrame(nb_r)
nb_r

,p_value,alpha
0,0.059802,0.669783
1,0.018272,0.174771
2,0.145331,0.436170
3,0.196138,0.194950


In [12]:
for i, row in nb_py.iterrows():
    r = nb_r.iloc[i]
    checks.add_equal(f"test chosen     {row['name']}", row.test, "Negative Binomial Regression (LRT)")
    checks.add(f"NB LRT p-value  {row['name']}", row.p_value, r.p_value, atol=1e-6, rtol=1e-3)
    checks.add(f"NB dispersion   {row['name']}", row.alpha, r.alpha, rtol=1e-3)

## 2b. Underdispersed counts: NB fit at the Poisson boundary

Items per order is always ≥ 1, and its variance is typically *below* its mean (e.g.
`1 + Poisson(0.5)` has variance/mean ≈ 0.33). NB2 can't fit underdispersion, so its
dispersion α goes to its lower bound and the NB model becomes Poisson. At that boundary
statsmodels can't invert the Hessian (it is singular in α alone) and returns NaN standard
errors, although the log-likelihoods and the LR statistic are fine. FOE then reports
`dispersion_alpha = 0.0` and checks the regression coefficients on a Poisson fit with the
same design.

Reference: the Poisson GLM LRT, `anova(glm(y ~ 1, poisson), glm(y ~ variant, poisson),
test = "LRT")`, checked for the global test and each pair against the control.
`MASS::glm.nb` is not a clean reference here: θ = 1/α runs off towards ∞ and it stops with
"iteration limit reached" warnings. Its θ and LR are recorded as a loose cross-check only.


In [13]:
import warnings
from statsmodels.tools.sm_exceptions import HessianInversionWarning

rng_ud = np.random.default_rng(2028)  # own stream: later sections keep their data
ud_sets = {
    "underdispersed, 2 groups": make_df({"A": 1 + rng_ud.poisson(0.5, 500),
                                         "B": 1 + rng_ud.poisson(0.8, 500)}),
    "underdispersed, 3 groups": make_df({"A": 1 + rng_ud.poisson(0.5, 500),
                                         "B": 1 + rng_ud.poisson(0.5, 500),
                                         "C": 1 + rng_ud.poisson(0.75, 500)}),
}
ud_py = {}
for name, df in ud_sets.items():
    with warnings.catch_warnings():  # the NaN-bse fit at the boundary is expected here
        warnings.simplefilter("ignore", HessianInversionWarning)
        res = engine.run_comparison_suite(df, ContinuousMetricConfig(
            kpi="value", group_col="variant", approach=ContinuousApproach.HEURISTIC,
            unit=AnalysisUnit.PER_TRANSACTION, control_label="A"))
    ud_py[name] = res
    print(f"{name}: var/mean = {df.value.var() / df.value.mean():.3f}, {res.test_name}, "
          f"LR = {res.test_statistic}, p = {res.p_value:.6g}, alpha = {res.dispersion_alpha}, "
          f"warnings = {res.warnings}, post-hoc = "
          f"{[(p.comparison, round(p.p_value, 6)) for p in (res.posthoc_results or [])]}")


underdispersed, 2 groups: var/mean = 0.412, Negative Binomial Regression (LRT), LR = 13.34161883613433, p = 0.00025958, alpha = 0.0, warnings = [], post-hoc = []
underdispersed, 3 groups: var/mean = 0.364, Negative Binomial Regression (LRT), LR = 8.243826473138142, p = 0.0162135, alpha = 0.0, warnings = [], post-hoc = [('B vs A', 0.85605), ('C vs A', 0.010355)]


In [14]:
%%R
suppressPackageStartupMessages(library(MASS))

poisson_lrt <- function(df) {
  m0 <- glm(value ~ 1, family = poisson, data = df)
  m1 <- glm(value ~ variant, family = poisson, data = df)
  a <- anova(m0, m1, test = "LRT")
  c(lr = a$Deviance[2], p_value = a[["Pr(>Chi)"]][2])
}

underdispersed_r <- function(df, control = "A") {
  df$variant <- factor(df$variant)
  g <- poisson_lrt(df)
  # glm.nb, for the record: it warns ("iteration limit reached") as theta runs off.
  n_warn <- 0
  nb <- withCallingHandlers({
    m1 <- glm.nb(value ~ variant, data = df); m0 <- glm.nb(value ~ 1, data = df)
    c(theta = m1$theta, lr = 2 * (as.numeric(logLik(m1)) - as.numeric(logLik(m0))))
  }, warning = function(w) { n_warn <<- n_warn + 1; invokeRestart("muffleWarning") })
  others <- setdiff(levels(df$variant), control)
  pairs <- t(sapply(others, function(v) {
    pdf <- droplevels(df[df$variant %in% c(control, v), ])
    poisson_lrt(pdf)
  }))
  data.frame(term = c("global", paste(others, "vs", control)),
             lr = c(g[["lr"]], pairs[, "lr"]), p_value = c(g[["p_value"]], pairs[, "p_value"]),
             nb_theta = nb[["theta"]], nb_lr = nb[["lr"]], nb_warnings = n_warn)
}


In [15]:
ud_r = {}
for name, df in ud_sets.items():
    %R -i df -o out out <- underdispersed_r(df)
    ud_r[name] = out.set_index("term")
    display(out)


,term,lr,p_value,nb_theta,nb_lr,nb_warnings
1,global,13.341656,0.00026,73921.77853,13.341994,4.0
2,B vs A,13.341656,0.00026,73921.77853,13.341994,4.0


,term,lr,p_value,nb_theta,nb_lr,nb_warnings
,global,8.243850,0.016213,77928.407626,8.244181,4.0
B,B vs A,0.032908,0.856049,77928.407626,8.244181,4.0
C,C vs A,6.572729,0.010355,77928.407626,8.244181,4.0


In [16]:
for name, res in ud_py.items():
    r = ud_r[name]
    checks.add_equal(f"test chosen         {name}", res.test_name, "Negative Binomial Regression (LRT)")
    checks.add_equal(f"no 'could not be fit' warning  {name}",
                     any("could not be fit" in w for w in res.warnings), False)
    checks.add_equal(f"NB dispersion = 0 (boundary)  {name}", res.dispersion_alpha, 0.0)
    checks.add(f"LR vs Poisson LRT   {name}", res.test_statistic, r.lr["global"], rtol=1e-5)
    checks.add(f"p vs Poisson LRT    {name}", res.p_value, r.p_value["global"], atol=1e-12, rtol=1e-4)
    checks.add(f"LR vs glm.nb (loose) {name}", res.test_statistic, r.nb_lr["global"], rtol=1e-3)
    checks.add_equal(f"glm.nb theta > 1e3 (alpha at bound)  {name}", bool(r.nb_theta["global"] > 1e3), True)
    for ph in (res.posthoc_results or []):
        checks.add(f"post-hoc p vs Poisson LRT  {name} {ph.comparison}",
                   ph.p_value, r.p_value[ph.comparison], atol=1e-12, rtol=1e-4)

# The 3-group set must exercise the post-hoc path.
checks.add_equal("post-hoc ran on underdispersed, 3 groups",
                 len(ud_py["underdispersed, 3 groups"].posthoc_results or []), 2)


True

## 3. Gamma family: per-transaction Gamma LRT and per-visitor hurdle LRT

In [17]:
gamma_tx = make_df({"A": rng.gamma(2.0, 30.0, 400), "B": rng.gamma(2.0, 34.0, 420),
                    "C": rng.gamma(2.4, 25.0, 410)})

# Per-visitor: ~4% of visitors buy; non-buyers are 0.
def rpv(n, p, k, theta):
    return np.where(rng.random(n) < p, rng.gamma(k, theta, n), 0.0)
gamma_pv = make_df({"A": rpv(6000, 0.040, 2.0, 30.0), "B": rpv(6000, 0.046, 2.0, 30.0),
                    "C": rpv(6000, 0.041, 2.0, 33.0)})

gamma_py = {}
for unit, df in [(AnalysisUnit.PER_TRANSACTION, gamma_tx), (AnalysisUnit.PER_VISITOR, gamma_pv)]:
    res = engine.run_comparison_suite(df, ContinuousMetricConfig(
        kpi="value", group_col="variant", approach=ContinuousApproach.GAMMA_GLM,
        unit=unit, control_label="A"))
    gamma_py[unit.value] = res
    print(unit.value, res.test_name, res.p_value,
          [(p.comparison, p.p_value) for p in (res.posthoc_results or [])])

per_transaction Gamma GLM (Likelihood Ratio Test) 0.003573641157954749 [('B vs A', 0.0015848660667450477), ('C vs A', 0.1133087633829012)]
per_visitor Two-Part Hurdle (Likelihood Ratio Test) 9.476171526094972e-06 [('B vs A', 0.0001274477984158504), ('C vs A', 0.0003875547195585593)]


In [18]:
%%R
# Exact Gamma MLE: the shape solves log(k) - digamma(k) = log(mean(x)) - mean(log(x)),
# then scale = mean(x) / k. Used instead of a general-purpose optimiser because LRT p-values
# depend on small log-likelihood differences.
gamma_mle <- function(x) {
  s <- log(mean(x)) - mean(log(x))
  k <- uniroot(function(k) log(k) - digamma(k) - s, c(1e-8, 1e8), tol = 1e-14)$root
  theta <- mean(x) / k
  list(k = k, theta = theta, ll = sum(dgamma(x, shape = k, scale = theta, log = TRUE)))
}
gamma_ll <- function(x) gamma_mle(x[x > 0])$ll
unit_ll <- function(x, unit) {
  if (unit == "per_transaction") return(gamma_ll(x))
  n <- length(x); k <- sum(x > 0); p <- k / n
  bern <- if (p > 0 && p < 1) k * log(p) + (n - k) * log(1 - p) else 0
  bern + gamma_ll(x)
}
gamma_lrt <- function(df, unit, control = "A") {
  npar <- if (unit == "per_transaction") 2 else 3
  if (unit == "per_transaction") df <- df[df$value > 0, ]
  groups <- split(df$value, df$variant)
  lr <- 2 * (sum(sapply(groups, unit_ll, unit = unit)) - unit_ll(df$value, unit))
  global <- pchisq(max(lr, 0), df = npar * (length(groups) - 1), lower.tail = FALSE)
  others <- setdiff(names(groups), control)
  post <- sapply(others, function(v) {
    lr <- 2 * (unit_ll(groups[[control]], unit) + unit_ll(groups[[v]], unit) -
               unit_ll(c(groups[[control]], groups[[v]]), unit))
    pchisq(max(lr, 0), df = npar, lower.tail = FALSE)
  })
  data.frame(term = c("global", paste(others, "vs", control)), p_value = c(global, post))
}

In [19]:
for unit, df in [("per_transaction", gamma_tx), ("per_visitor", gamma_pv)]:
    %R -i df -i unit -o out out <- gamma_lrt(df, unit)
    res = gamma_py[unit]
    checks.add(f"Gamma LRT global p  {unit}", res.p_value, out.p_value.iloc[0], atol=1e-12, rtol=1e-7)
    for ph in (res.posthoc_results or []):
        r_p = out.set_index("term").p_value[ph.comparison]
        checks.add(f"Gamma post-hoc p    {unit} {ph.comparison}", ph.p_value, r_p, atol=1e-12, rtol=1e-7)
    display(out)

,term,p_value
,global,0.003574
B,B vs A,0.001585
C,C vs A,0.113309


,term,p_value
,global,0.000009
B,B vs A,0.000127
C,C vs A,0.000388


## 4. Distribution fits used by the pre-test planner

In [20]:
x_gamma = rng.gamma(1.8, 40.0, 2000)
k, theta, ll = ContinuousMetricEngine.fit_gamma(x_gamma)
x_counts = nb_draw(2.5, 0.7, 2000).astype(float)
nb_ll, nb_alpha, nb_res = ContinuousMetricEngine.fit_negbin(x_counts)
nb_mean = float(np.exp(nb_res.params[0]))

In [21]:
%%R -i x_gamma -i x_counts -o fits_r
g <- gamma_mle(x_gamma)
g_fd <- fitdistr(x_gamma, "gamma", start = list(shape = 1, rate = 1 / mean(x_gamma)), lower = c(1e-8, 1e-10))
nb <- fitdistr(x_counts, "negative binomial")
fits_r <- data.frame(k = g$k, theta = g$theta, ll = g$ll,
                     k_fitdistr = g_fd$estimate[["shape"]], theta_fitdistr = 1 / g_fd$estimate[["rate"]],
                     nb_mu = nb$estimate[["mu"]], nb_alpha = 1 / nb$estimate[["size"]], nb_ll = nb$loglik)
fits_r

       k

   theta

        ll

 k_fitdistr

 theta_fitdistr

    nb_mu

... (8 outputs left out)

 0.600637

     nb_ll


1

 -4086.221

In [22]:
f = fits_r.iloc[0]
checks.add("fit_gamma shape k", k, f.k, rtol=1e-10)
checks.add("fit_gamma shape k (vs MASS::fitdistr)", k, f.k_fitdistr, rtol=5e-3)
checks.add("fit_gamma scale theta", theta, f.theta, rtol=1e-10)
checks.add("fit_gamma log-likelihood", ll, f.ll, rtol=1e-12)
checks.add("fit_negbin mean", nb_mean, f.nb_mu, rtol=1e-4)
checks.add("fit_negbin alpha", nb_alpha, f.nb_alpha, rtol=1e-3)
checks.add("fit_negbin log-likelihood", nb_ll, f.nb_ll, rtol=1e-8)

True

## 5. Winsorizing

In [23]:
w = pd.Series(rng.lognormal(3.5, 1.0, 3000))
_, sd_lo, sd_hi = ContinuousMetricEngine.winsorize_series(w, "Standard Deviation", 3.0)
_, pc_lo, pc_hi = ContinuousMetricEngine.winsorize_series(w, "Percentile", 1.0)
w = w.to_numpy()

# Per-visitor shape: 40% zeros. Caps come from the non-zero values only.
rng_w = np.random.default_rng(2028)  # own stream
wz = pd.Series(np.r_[np.zeros(800), rng_w.lognormal(3.5, 1.0, 1200)])
wz_capped, z_sd_lo, z_sd_hi = ContinuousMetricEngine.winsorize_series(
    wz, "Standard Deviation", 3.0, exclude_zeros=True)
_, z_cp_lo, z_cp_hi = ContinuousMetricEngine.winsorize_series(
    wz, "Central Percentile", 95.0, exclude_zeros=True)
zeros_untouched = bool(np.all(np.asarray(wz_capped)[:800] == 0))
wz = wz.to_numpy()

In [24]:
%%R -i w -i wz -o wins_r
nz <- wz[wz != 0]
wins_r <- c(mean(w) - 3 * sd(w), mean(w) + 3 * sd(w), quantile(w, c(0.01, 0.99), type = 7),
            mean(nz) - 3 * sd(nz), mean(nz) + 3 * sd(nz), quantile(nz, c(0.025, 0.975), type = 7))

In [25]:
checks.add("winsorize SD lower", sd_lo, wins_r[0], rtol=1e-10)
checks.add("winsorize SD upper", sd_hi, wins_r[1], rtol=1e-10)
checks.add("winsorize P1 lower", pc_lo, wins_r[2], rtol=1e-10)
checks.add("winsorize P99 upper", pc_hi, wins_r[3], rtol=1e-10)
checks.add("winsorize SD lower, zeros excluded", z_sd_lo, wins_r[4], rtol=1e-10)
checks.add("winsorize SD upper, zeros excluded", z_sd_hi, wins_r[5], rtol=1e-10)
checks.add("winsorize central 95% lower, zeros excluded", z_cp_lo, wins_r[6], rtol=1e-10)
checks.add("winsorize central 95% upper, zeros excluded", z_cp_hi, wins_r[7], rtol=1e-10)
checks.add_equal("winsorize leaves zero rows at 0", zeros_untouched, True)

True

## 5b. Outlier detection (`detect_outliers`)

Zero rows are never considered. Up to `large_n_threshold` non-zero rows, a row is flagged when
|internally studentized residual|, leverage or |DFFITS| of `lm(value ~ variant)` passes its
threshold (k, k·p/n, k·√(p/n)). Above it, a row is flagged when it lies outside mean ± k·sd of
its own variant. R recomputes both masks with `rstandard`, `hatvalues` and `dffits`.

In [26]:
rng_o = np.random.default_rng(2029)  # own stream


def spiky(n, p_buy, n_spikes):
    x = np.where(rng_o.random(n) < p_buy, rng_o.lognormal(3.5, 0.6, n), 0.0)
    idx = rng_o.choice(np.flatnonzero(x), n_spikes, replace=False)
    x[idx] *= rng_o.uniform(4, 10, n_spikes)
    return x

out_sets = {
    "OLS influence, 3 variants": (make_df({"A": spiky(900, 0.3, 4), "B": spiky(900, 0.3, 3),
                                           "C": spiky(900, 0.3, 5)}), 10_000),
    "per-variant SD (forced large)": (make_df({"A": spiky(900, 0.3, 4),
                                               "B": spiky(900, 0.3, 3)}), 100),
}
out_py = {}
for name, (df, large) in out_sets.items():
    res = ContinuousMetricEngine.detect_outliers(df, "value", group_col="variant",
                                                 threshold=3.0, large_n_threshold=large)
    out_py[name] = res
    print(f"{name}: method={res.method}, considered={res.n_considered}, "
          f"flagged={res.n_outliers}")

OLS influence, 3 variants: method=ols_influence, considered=822, flagged=10
per-variant SD (forced large): method=per_variant_sd, considered=476, flagged=6


In [27]:
%%R
outliers_r <- function(df, thr, large) {
  keep <- !is.na(df$value) & df$value != 0
  sub <- df[keep, ]; sub$variant <- factor(sub$variant)
  if (nrow(sub) > large) {
    flag <- ave(sub$value, sub$variant, FUN = function(v) abs(v - mean(v)) > thr * sd(v)) == 1
  } else {
    fit <- lm(value ~ variant, data = sub)
    p <- length(coef(fit)); n <- nrow(sub)
    flag <- abs(rstandard(fit)) > thr | hatvalues(fit) > thr * p / n |
            abs(dffits(fit)) > thr * sqrt(p / n)
  }
  mask <- rep(FALSE, nrow(df)); mask[keep] <- as.logical(flag)
  mask
}

In [28]:
for name, (df, large) in out_sets.items():
    large = float(large)
    %R -i df -i large -o mask_r mask_r <- outliers_r(df, 3, large)
    mask_py = np.array(out_py[name].mask)
    mask_r = np.asarray(mask_r, dtype=bool)
    checks.add_equal(f"outlier mask mismatches  {name}", int((mask_py != mask_r).sum()), 0)
    checks.add_equal(f"outliers flagged         {name}", int(mask_py.sum()), int(mask_r.sum()))
    checks.add_equal(f"no zero row flagged      {name}",
                     bool(mask_py[df.value.to_numpy() == 0].any()), False)

## 6. Business case (`run_business_case`)

The projected value per visitor is `rate · mean` per arm; its SE comes from the delta method.
Per visitor, rate = 1; per transaction, rate = orders / visitors with a binomial SE.

In [29]:
group_stats = {"A": {"mean": 61.2, "std": 48.0, "count": 1450},
               "B": {"mean": 63.9, "std": 50.5, "count": 1510}}
visitor_counts = {"A": 40_000, "B": 40_200}
daily_visitors, period = 5_000, 183
bc = {}
for unit in [AnalysisUnit.PER_VISITOR, AnalysisUnit.PER_TRANSACTION]:
    bc[unit.value] = engine.run_business_case(
        group_stats=group_stats, control_label="A", unit=unit, daily_visitors=daily_visitors,
        visitor_counts=visitor_counts, alternative=AlternativeHypothesis.TWO_SIDED,
        projection_period=period)[0]
m_a, s_a, n_a = 61.2, 48.0, 1450
m_b, s_b, n_b = 63.9, 50.5, 1510
v_a, v_b = 40_000, 40_200
pd.DataFrame(bc).loc[["point_estimate", "ci_low", "ci_high"]]

,per_visitor,per_transaction
point_estimate,2470500.0,166277.350746
ci_low,-776372.489658,-25320.583933
ci_high,5717372.489658,357875.285425


In [30]:
%%R -i m_a -i s_a -i n_a -i m_b -i s_b -i n_b -i v_a -i v_b -i daily_visitors -i period -o bc_r
k <- daily_visitors * period
z <- qnorm(0.975)
# per visitor: X = m_b - m_a
se_pv <- deltamethod(~ x1 - x2, c(m_b, m_a), diag(c(s_b^2 / n_b, s_a^2 / n_a)))
x_pv <- m_b - m_a
# per transaction: X = r_b * m_b - r_a * m_a, r = orders / visitors
r_a <- n_a / v_a; r_b <- n_b / v_b
mu <- c(r_b, m_b, r_a, m_a)
V <- diag(c(r_b * (1 - r_b) / v_b, s_b^2 / n_b, r_a * (1 - r_a) / v_a, s_a^2 / n_a))
se_pt <- deltamethod(~ x1 * x2 - x3 * x4, mu, V)
x_pt <- r_b * m_b - r_a * m_a
bc_r <- data.frame(unit = c("per_visitor", "per_transaction"),
                   point_estimate = c(x_pv, x_pt) * k,
                   ci_low = (c(x_pv, x_pt) - z * c(se_pv, se_pt)) * k,
                   ci_high = (c(x_pv, x_pt) + z * c(se_pv, se_pt)) * k)
bc_r

            unit

 point_estimate

     ci_low

   ci_high


1

     per_visitor

... (4 outputs left out)

 per_transaction

       166277.4

  -25320.58

  357875.3

In [31]:
for _, r in bc_r.iterrows():
    for key in ["point_estimate", "ci_low", "ci_high"]:
        checks.add(f"business case {key} | {r.unit}", bc[r.unit][key], r[key], rtol=1e-8)

## Summary

In [32]:
checks.report()

271 checks, 271 passed, 0 failed

Largest differences:
                                        check   hexkit        R    abs_diff              tolerance
        fit_gamma shape k (vs MASS::fitdistr)  1.76066  1.76265  0.00198923 atol=1e-08, rtol=0.005
LR vs glm.nb (loose) underdispersed, 2 groups  13.3416   13.342 0.000375027 atol=1e-08, rtol=0.001
LR vs glm.nb (loose) underdispersed, 3 groups  8.24383  8.24418 0.000354092 atol=1e-08, rtol=0.001
 LR vs Poisson LRT   underdispersed, 2 groups  13.3416  13.3417   3.732e-05 atol=1e-08, rtol=1e-05
                             fit_negbin alpha 0.600603 0.600637 3.38266e-05 atol=1e-08, rtol=0.001


,check,hexkit,R,abs_diff,tolerance,pass
0,"is_normal normal, equal var (ANOVA)",True,True,NaN,exact,True
1,"is_homogeneous normal, equal var (ANOVA)",True,True,NaN,exact,True
2,"test chosen normal, equal var (ANOVA)",Standard ANOVA,Standard ANOVA,NaN,exact,True
3,"p-value normal, equal var (ANOVA)",0.009942,0.009942,2.163200e-15,"atol=1e-10, rtol=1e-06",True
4,"statistic normal, equal var (ANOVA)",4.628834,4.628834,2.184919e-13,"atol=1e-08, rtol=1e-08",True
...,...,...,...,...,...,...
266,business case ci_low | per_visitor,-776372.489658,-776372.489658,3.492460e-10,"atol=1e-08, rtol=1e-08",True
267,business case ci_high | per_visitor,5717372.489658,5717372.489658,9.313226e-10,"atol=1e-08, rtol=1e-08",True
268,business case point_estimate | per_transaction,166277.350746,166277.350746,0.000000e+00,"atol=1e-08, rtol=1e-08",True
269,business case ci_low | per_transaction,-25320.583933,-25320.583933,2.910383e-11,"atol=1e-08, rtol=1e-08",True
